# Importing Libraries

In [1]:
## Import modules
import os, sys
import numpy as np
import geopandas as gpd
import cftime
import gc
import shapely
import json
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
import matplotlib.pyplot as plt
# Import Plotly for interactive plotting
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as pc
from ipywidgets import interact, IntSlider, Dropdown, VBox, HBox
import ipywidgets as widgets

# Add the directory containing 'cmct' to the Python path
# Navigate two levels up to reach main CmCt dir
cmct_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))

# Initialising Logger
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Import utilities for this comparison
sys.path.insert(0, cmct_dir)
from cmct.time_utils import *
from cmct.calving import *
from cmct.calving_modules.interpolation import *
from cmct.calving_modules.residual_calculation import *
# from cmct.calving_modules.json_to_netcdf import *
# from cmct.shapefile_utils import *

# Force initial garbage collection
gc.collect()

40

In [2]:
# Reload modules to pick up any changes to imports
import importlib
import cmct.calving
import cmct.calving_modules.residual_calculation
import cmct.calving_modules.plotting_utils
from cmct.calving_modules.plotting_utils import *
from cmct.calving import calculate_basin_statistics, format_basin_stats
from cmct.calving import calculate_basin_statistics
importlib.reload(cmct.calving)
importlib.reload(cmct.calving_modules.residual_calculation)
importlib.reload(cmct.calving_modules.plotting_utils)

# Re-import to ensure functions are available
from cmct.calving import *

# CONFIGURATION

In [3]:
# Observation Dataset
# Ice sheet
loc = "GIS"  # 'GIS' or 'AIS'

# Set the observation data dir path
obs_filename = cmct_dir + "/data/calving/observed_icemask_ismip_annual.nc"

# To use aggregation functions for basin
basin_aggregation = True  # IMPORTANT

basin_filename = cmct_dir + "/bin/Calving/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp"

# Set the Model Data dir path
# model_filename = cmct_dir + "/test/calving/ensemble/sftgif_B001_hist.nc"
model_filename = cmct_dir + "/test/calving/sftgif_GIS_JPL_ISSM_historical.nc"

# Set time range for comparison
start_year = 2006
end_year = 2015

# List of basins (ex ["NW", "NE"]) to compare if all -> "all", if none -> False
# If you do not know which basins are in the model, you can put "auto"
# NOTE: Align this list with the basins in the model.
basin_list = "all"

# Output filetype and filename
filetype = "netcdf"  # netcdf or json or None
filename = "calving_comparison"

# Optional Configurations
interpolation_method = "slinear"  # 'nearest', 'linear', 'cubic'
accuracy_calculation_method = "mean"  # 'mean', 'RMS',

colors = {
    "CW": "blue",
    "NE": "red",
    "SE": "green",
    "SW": "orange",
    "NO": "purple",
    "NW": "brown",
}


# Loading all data files

In [4]:
# Check if observation file exist
if not os.path.exists(obs_filename):
    raise FileNotFoundError(f"Observation file not found: {obs_filename}")

# # Check if model file exist
if not os.path.exists(model_filename):
    raise FileNotFoundError(f"Model file not found: {model_filename}")



if basin_aggregation and not os.path.exists(basin_filename):
    raise FileNotFoundError(f"Basin shapefile not found: {basin_filename}")
    # Load basin shapes

print(basin_filename)
basins, basin_list = load_basins(basin_filename, basin_list)

print(obs_filename)
gsfc = load_gsfc_calving(obs_filename, basins)

print(model_filename)
model_res = load_model_calving(model_filename)

/Users/aditya_pachpande/Documents/GitHub/CmCt/bin/Calving/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp
/Users/aditya_pachpande/Documents/GitHub/CmCt/data/calving/observed_icemask_ismip_annual.nc
/Users/aditya_pachpande/Documents/GitHub/CmCt/test/calving/sftgif_GIS_JPL_ISSM_historical.nc


## Handelling Time Consistency

In [5]:
# Simplifying date data type
gsfc.ds["time"] = standardising_time_var(gsfc.time)
model_res.ds["time"] = standardising_time_var(model_res.time)

# Handelling Time Range
checking_calving_daterange(gsfc.time.values, model_res.time.values, start_year, end_year)


The selected dates 2006 to 2015 are within the overlapping data range.


# Interpolation

In [6]:
interpolater = Interpolater(model_res, gsfc)
model_res.ds = interpolater.interpolate()

2025-07-21 12:54:51,546 - INFO - Input x coordinates: [-720000. -715000. -710000. -705000. -700000. -695000. -690000. -685000.
 -680000. -675000. -670000. -665000. -660000. -655000. -650000. -645000.
 -640000. -635000. -630000. -625000. -620000. -615000. -610000. -605000.
 -600000. -595000. -590000. -585000. -580000. -575000. -570000. -565000.
 -560000. -555000. -550000. -545000. -540000. -535000. -530000. -525000.
 -520000. -515000. -510000. -505000. -500000. -495000. -490000. -485000.
 -480000. -475000. -470000. -465000. -460000. -455000. -450000. -445000.
 -440000. -435000. -430000. -425000. -420000. -415000. -410000. -405000.
 -400000. -395000. -390000. -385000. -380000. -375000. -370000. -365000.
 -360000. -355000. -350000. -345000. -340000. -335000. -330000. -325000.
 -320000. -315000. -310000. -305000. -300000. -295000. -290000. -285000.
 -280000. -275000. -270000. -265000. -260000. -255000. -250000. -245000.
 -240000. -235000. -230000. -225000. -220000. -215000. -210000. -20500

# Comparison and Residual Calculation 

In [7]:
print(type(basins))
years = np.arange(start_year, end_year + 1)

residuals = create_calving_dataset(gsfc, model_res, years, basins)

2025-07-21 12:54:59,649 - INFO - Starting optimized calving dataset creation...
2025-07-21 12:54:59,659 - INFO - Transformed basin CW: 1211 points
2025-07-21 12:54:59,674 - INFO - Transformed basin NE: 6780 points
2025-07-21 12:54:59,693 - INFO - Transformed basin SE: 15619 points
2025-07-21 12:54:59,698 - INFO - Transformed basin SW: 4063 points
2025-07-21 12:54:59,703 - INFO - Transformed basin NO: 4016 points
2025-07-21 12:54:59,711 - INFO - Transformed basin NW: 4986 points
2025-07-21 12:54:59,727 - INFO - Grid dimensions: 2880 x 1680


<class 'dict'>


2025-07-21 12:55:00,240 - INFO - Computing residuals...
2025-07-21 12:55:00,493 - INFO - Computing statistics...
2025-07-21 12:55:00,518 - INFO - Creating basin assignments...
2025-07-21 12:55:00,519 - INFO - Data coordinates: X=[-719500.0, 959500.0], Y=[-3449500.0, -570500.0]
2025-07-21 12:55:00,519 - INFO - Basin polygon coordinates: X=[-607915.8, 839103.9], Y=[-3309066.0, -789966.1]
2025-07-21 12:55:28,625 - INFO - Basin assignment complete. Unique basin IDs: [-1  0  1  2  3  4  5]
2025-07-21 12:55:28,629 - INFO -   Unassigned points: 3112422
2025-07-21 12:55:28,632 - INFO -   Basin 0 (CW): 232302 points
2025-07-21 12:55:28,636 - INFO -   Basin 1 (NE): 478030 points
2025-07-21 12:55:28,639 - INFO -   Basin 2 (SE): 294627 points
2025-07-21 12:55:28,643 - INFO -   Basin 3 (SW): 217562 points
2025-07-21 12:55:28,650 - INFO -   Basin 4 (NO): 232587 points
2025-07-21 12:55:28,657 - INFO -   Basin 5 (NW): 270870 points
2025-07-21 12:55:28,669 - INFO - Basin assignment rate: 1725978/483840

In [8]:
residuals = load_residuals(residuals)


In [9]:
vars(residuals)


{'ds': <xarray.Dataset> Size: 774MB
 Dimensions:                 (time: 10, y: 2880, x: 1680, basin_id: 6)
 Coordinates:
   * time                    (time) int64 80B 2006 2007 2008 ... 2013 2014 2015
   * x                       (x) float32 7kB -7.195e+05 -7.185e+05 ... 9.595e+05
   * y                       (y) float32 12kB -3.45e+06 -3.448e+06 ... -5.705e+05
     basin_names             (basin_id) <U2 48B 'CW' 'NE' 'SE' 'SW' 'NO' 'NW'
 Dimensions without coordinates: basin_id
 Data variables:
     residual                (time, y, x) float32 194MB 0.0 0.0 0.0 ... 0.0 0.0
     basin                   (time, y, x) int32 194MB -1 -1 -1 -1 ... -1 -1 -1 -1
     gsfc_ice_mask           (time, y, x) float32 194MB 0.0 0.0 0.0 ... 0.0 0.0
     model_ice_mask          (time, y, x) float32 194MB 0.0 0.0 0.0 ... 0.0 0.0
     stats_avg_abs_residual  (time) float64 80B 0.02461 0.02462 ... 0.02482
     stats_rms_residual      (time) float64 80B 0.1319 0.1319 ... 0.1326 0.1327
     stats_sum_residu

# Statistics Calculation

In [10]:
basin_stats = calculate_basin_statistics(residuals)
print(format_basin_stats(basin_stats))

2025-07-21 12:55:28,816 - INFO - Starting basin statistics calculation
2025-07-21 12:55:28,817 - INFO - Processing 10 time steps and 6 basins
2025-07-21 12:55:29,973 - INFO - Basin statistics calculation completed


=== Statistics for Year 2006 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
CW    | 23.257492065429688 |   232302 |  0.00010012 |  0.00013375 |  0.00004944 |   0.036565 |   0.036565
NE    | -2797.58154296875 |   478030 | -0.00585231 | -0.00666378 | -0.00003793 |   0.115760 |   0.115908
SE    | 14.06719970703125 |   294627 |  0.00004775 | -0.00724717 | -0.00024722 |   0.204647 |   0.204647
SW    | 1017.2067260742188 |   217562 |  0.00467548 |  0.00493314 |  0.00028247 |   0.081600 |   0.081734
NO    | 2110.669189453125 |   232587 |  0.00907475 |  0.00970253 |  0.00030426 |   0.115014 |   0.115371
NW    | -181.97161865234375 |   270870 | -0.00067180 | -0.00062181 |  0.00009068 |   0.060439 |   0.060443
-------------------------------------------------------------------------------------


=== Statistics for Year 2007 ===
Basin | Sum       | Count    | Mean 

# Plot Generation

## Configuration
- residuals are required for plot generation

In [11]:
# Import plotting functions from plotting_utils
from cmct.calving_modules.plotting_utils import (
    create_time_series_plot,
    create_relative_time_series_plot,
    create_gsfc_model_residual_grid,
    create_correlation_matrix,
)

# Which year do you want plots for?
year = 2007

cmap = "ocean"  # Leave default to Ocean :)
aspect = "auto"

# Plot config
plt.figure(figsize=(10, 6))
basin_id = 0
# plt.cm.viridis.set_bad('lightblue')  # Set color for NaN values

<Figure size 1000x600 with 0 Axes>

In [12]:
create_interactive_residual_plot(residuals, year, basin_id=basin_id)
    
create_basin_statistics_plot(basin_stats, year)
  
# Create interactive widgets
year_slider = IntSlider(
    value=start_year,
    min=start_year,
    max=end_year,
    step=1,
    description="Year:",
    continuous_update=False
)

basin_options = [('All Basins', -1)] + [(name, idx) for idx, name in enumerate(residuals.ds.basin_names.values)]
basin_dropdown = Dropdown(
    options=basin_options,
    value=-1,
    description="Basin:",
)

plot_type_dropdown = Dropdown(
    options=[('Residual Map', 'residual'), ('Basin Statistics', 'stats')],
    value='residual',
    description="Plot Type:"
)

# Define the corrected interactive plot function
def interactive_plot(year, basin_id, plot_type):
    """Enhanced interactive plotting function with multiple plot types"""
    
    if plot_type == "residual":
        basin_id = None if basin_id == -1 else int(basin_id)
        fig = create_interactive_residual_plot(residuals, year, basin_id)
        if fig:
            fig.show()
    
    elif plot_type == "stats":
        fig = create_basin_statistics_plot(basin_stats, year)
        if fig:
            fig.show()

# Create the interactive widget
interact(interactive_plot, 
         year=year_slider, 
         basin_id=basin_dropdown, 
         plot_type=plot_type_dropdown)

interactive(children=(IntSlider(value=2006, continuous_update=False, description='Year:', max=2015, min=2006),…

<function __main__.interactive_plot(year, basin_id, plot_type)>

# Interactive Plotting Analysisdd

In [13]:
statistic_dropdown = Dropdown(
    options=[('Mean', 'mean'), ('Standard Deviation', 'std'), ('RMS', 'rms'), 
             ('Winsorized Mean', 'winsorized_mean'), ('Outlier Weighted Mean', 'outlier_weighted_mean'), ('Sum', 'sum')],
    value='mean',
    description="Statistic:"
)

def interactive_grid_plot(statistic):
    """Interactive grid plotting function"""
    fig = create_gsfc_model_residual_grid(basin_stats, statistic, colors=colors)
    if fig:
        fig.show()

interact(interactive_grid_plot, statistic=statistic_dropdown)

interactive(children=(Dropdown(description='Statistic:', options=(('Mean', 'mean'), ('Standard Deviation', 'st…

<function __main__.interactive_grid_plot(statistic)>

In [14]:
statistic_dropdown_rel = Dropdown(
    options=[('Mean', 'mean'), ('Standard Deviation', 'std'), ('RMS', 'rms'), 
             ('Winsorized Mean', 'winsorized_mean'), ('Outlier Weighted Mean', 'outlier_weighted_mean'), ('Sum', 'sum')],
    value='mean',
    description="Statistic:"
)

def interactive_relative_time_series(statistic):
    """Interactive relative time series plotting"""
    fig = create_relative_time_series_plot(basin_stats, statistic, colors=colors)
    if fig:
        fig.show()

interact(interactive_relative_time_series, statistic=statistic_dropdown_rel)

interactive(children=(Dropdown(description='Statistic:', options=(('Mean', 'mean'), ('Standard Deviation', 'st…

<function __main__.interactive_relative_time_series(statistic)>

# Correlation Analysis

In [15]:
def interactive_correlation(year):
    """Interactive correlation matrix"""
    fig = create_correlation_matrix(basin_stats, year)
    if fig:
        fig.show()
    else:
        print(f"Cannot create correlation matrix for year {year}")

year_slider_corr = IntSlider(
    value=start_year,
    min=start_year,
    max=end_year,
    step=1,
    description="Year:",
    continuous_update=False,
)

interact(interactive_correlation, year=year_slider_corr)

interactive(children=(IntSlider(value=2006, continuous_update=False, description='Year:', max=2015, min=2006),…

<function __main__.interactive_correlation(year)>